In [ ]:
# small library of exercises for intial testing and buulding will expand for full app
exercise_library = [
    {"name": "Back Squat", "category": "Strength", "movement_pattern": "Squat", "equipment": "Barbell", "muscle_group": "Quads"},
    {"name": "Bench Press", "category": "Strength", "movement_pattern": "Horizontal Push", "equipment": "Barbell", "muscle_group": "Chest"},
    {"name": "Deadlift", "category": "Strength", "movement_pattern": "Hinge", "equipment": "Barbell", "muscle_group": "Hamstrings"},
    {"name": "Overhead Press", "category": "Strength", "movement_pattern": "Vertical Push", "equipment": "Barbell", "muscle_group": "Shoulders"},
    {"name": "Pull-Up", "category": "Bodyweight", "movement_pattern": "Vertical Pull", "equipment": "Bodyweight", "muscle_group": "Back"},
]

print(f"{len(exercise_library)} exercises loaded")
print([ex["name"] for ex in exercise_library])

5 exercises loaded
['Back Squat', 'Bench Press', 'Deadlift', 'Overhead Press', 'Pull-Up']


In [ ]:
import pandas as pd


workout_log = pd.DataFrame({ #create empty dataframe with columns and data types
    # "workout_id": pd.Series(dtype='object'),   # stretch: unique ID per workout will enable multiple sessions/day + real-time logging (V1 uses date instead)
    'date': pd.Series(dtype='datetime64[us]'),
    'exercise_name': pd.Series(dtype='object'),
    'set_number': pd.Series(dtype='int'),
    'reps': pd.Series(dtype='int'),
    'weight': pd.Series(dtype='float'), 
    'notes': pd.Series(dtype='object')
})  

# prnt the data type of each column
print(workout_log.dtypes)

workout_log

date             datetime64[ns]
exercise_name            object
set_number                int64
reps                      int64
weight                  float64
notes                    object
dtype: object


,date,exercise_name,set_number,reps,weight,notes


In [ ]:
def build_set(date, exercise_name, set_number, reps, weight, notes=""): 
    """Build a single set's row as a dict. The atom the other functions assemble."""
    return {
        "date": date,
        "exercise_name": exercise_name,
        "set_number": set_number,
        "reps": reps,
        "weight": weight,
        "notes": notes,
    }

# testing
build_set("2026-06-21", "Back Squat", 1, 5, 185.0, "felt strong")

{'date': '2026-06-21',
 'exercise_name': 'Back Squat',
 'set_number': 1,
 'reps': 5,
 'weight': 185.0,
 'notes': 'felt strong'}

In [ ]:
#testing no notes and body weight use case 0.0 for weight
build_set("2026-06-21", "Pull-Up", 1, 8, 0.0)

{'date': '2026-06-21',
 'exercise_name': 'Pull-Up',
 'set_number': 1,
 'reps': 8,
 'weight': 0.0,
 'notes': ''}

In [ ]:
# log_exercise function to build all rows for one exercise in one day it takes a date, exercise name, and a list of sets and returns a list of rows (dicts) that we will add to the dataframe
def log_exercise(date, exercise_name, sets): 
    """
    Build all rows for one exercise in one day.
    `sets` is a list of (reps, weight, notes) tuples one per set.
    set_number is assigned automatically, starting at 1.
    """

    rows = [] #empty list to store set rows
    set_number = 1 #start at 1
    for this_set in sets: #iterate over the list of sets
        reps, weight, notes = this_set #unpack the tuple
        rows.append(build_set(date, exercise_name, set_number, reps, weight, notes)) #add the row to the list
        set_number = set_number + 1  # add 1 to the counter for the next set
    return rows

# testing
log_exercise("2026-06-21", "Back Squat", [(5, 185.0, "felt strong"), (5, 185.0, ""), (3, 190.0, "grinder")])

[{'date': '2026-06-21',
  'exercise_name': 'Back Squat',
  'set_number': 1,
  'reps': 5,
  'weight': 185.0,
  'notes': 'felt strong'},
 {'date': '2026-06-21',
  'exercise_name': 'Back Squat',
  'set_number': 2,
  'reps': 5,
  'weight': 185.0,
  'notes': ''},
 {'date': '2026-06-21',
  'exercise_name': 'Back Squat',
  'set_number': 3,
  'reps': 3,
  'weight': 190.0,
  'notes': 'grinder'}]

In [ ]:
def log_workout(df, date, exercises): 
    """
    Add a full workout to the log dataframe.
    `exercises` is a list of (exercise_name, sets) pairs.
    Collects every set-dict into one batch, then concats onto dataframe in one go.
    Returns the new, larger dataframe (does not modify df in place).
    """
    new_workout = [] #empty list to store new workout rows
    for exercise_name, sets in exercises: #iterate over the list of exercises and sets
        new_workout.extend(log_exercise(date, exercise_name, sets)) #add the rows to the new workout list

    new_rows = pd.DataFrame(new_workout) #create a new dataframe from the new workout list
    return pd.concat([df, new_rows], ignore_index=True) #concatenate the new dataframe to the existing dataframe and return the result

In [37]:
#testing
test_workout = [
    ("Back Squat", [(5, 185.0, "felt strong"), (5, 185.0, ""), (3, 190.0, "grinder")]),
    ("Bench Press", [(8, 135.0, "good speed")]),
    ("Pull-Up", [(8, 0.0, ""), (6, 0.0, "")]),
]

workout_log = log_workout(workout_log, "2026-06-21", test_workout)
workout_log

,date,exercise_name,set_number,reps,weight,notes
0,2026-06-21,Back Squat,1,5,185.0,felt strong
1,2026-06-21,Back Squat,2,5,185.0,
2,2026-06-21,Back Squat,3,3,190.0,grinder
3,2026-06-21,Bench Press,1,8,135.0,good speed
4,2026-06-21,Pull-Up,1,8,0.0,
5,2026-06-21,Pull-Up,2,6,0.0,


In [38]:
def save_log(df, path="workout_log.csv"): 
    """Write the log frame to CSV. index=False keeps pandas' row index out of the file."""
    df.to_csv(path, index=False) #write the dataframe to a csv file

# testing
save_log(workout_log)

In [42]:
import os

def load_log(path="workout_log.csv"):
    """
    Read the log CSV back into a typed frame.
    parse_dates rebuilds the datetime type (fixing the logging-time drift).
    """
    return pd.read_csv(path, parse_dates=["date"]) #read the csv file back into a dataframe and parse the date column to a datetime object

# test line — comment out once confirmed
loaded = load_log()
print(loaded.dtypes)
loaded

date             datetime64[us]
exercise_name               str
set_number                int64
reps                      int64
weight                  float64
notes                       str
dtype: object


,date,exercise_name,set_number,reps,weight,notes
0,2026-06-21,Back Squat,1,5,185.0,felt strong
1,2026-06-21,Back Squat,2,5,185.0,NaN
2,2026-06-21,Back Squat,3,3,190.0,grinder
3,2026-06-21,Bench Press,1,8,135.0,good speed
4,2026-06-21,Pull-Up,1,8,0.0,NaN
5,2026-06-21,Pull-Up,2,6,0.0,NaN
